# Job Hunting Aggregator

Scrape all jobs from: https://ph.indeed.com/?r=us, https://ph.jobstreet.com/, https://www.linkedin.com/

In [13]:
from bs4 import BeautifulSoup
from seleniumbase import SB
import pandas as pd

In [ ]:
def get_url (indeed_url, job_title, location, job_type = ''):
    template = '{}/jobs?q={}&l={}&sc={}'
    url = template.format(indeed_url,job_title, location, job_type)
    return url

In [15]:
df = pd.DataFrame({'Job Title': [''],  'Company Name': [''], 'Location': [''], 'Salary Range': [''], 'Summary': [''], 'Link': ['']})

In [19]:
indeed_url = "https://ph.indeed.com"
title = 'python'
location = 'philippines'
# filter = '0kf%3Aattr%287EQCZ%29%3B'
filter = ''
url = get_url(indeed_url, title, location, filter)

with SB(uc_cdp=True, incognito=True) as sb:
    while True: # Loop until next button is gone
        while True:  # Loop until the text is found
            sb.open(url)
            sb.uc_gui_click_captcha()
            sb.sleep(20) # Adjust values depending on load time

            try:
                sb.assert_text("Find jobs", "button.yosegi-InlineWhatWhere-primaryButton[type='submit']")
                print("Text Found")
                break  # Exit loop if text is found
            except Exception:
                print("Text not found. Retrying...")
                continue  # Retry from sb.open(url)

        raw_html = sb.get_page_source()
        soup = BeautifulSoup(raw_html, 'lxml')
        cards = soup.find_all('div', 'job_seen_beacon')

        for card in cards:
            td_tag = card.table.tbody.tr.td

            # Job Title and URL
            try:
                job_div = td_tag.find('div', {'class','css-pt3vth e37uo190'}).h2.a
                job_title = job_div.span.get('title')
            except AttributeError:
                job_title = None
            try:
                job_url = indeed_url + job_div.get('href')
            except Exception:
                job_url = None

            # Company Name
            try:
                company_div = td_tag.find('div', {'class','company_location css-i375s1 e37uo190'})
                company_name = company_div.find('div', {'class','css-1afmp4o e37uo190'}).span.text.strip()
            except AttributeError:
                company_name = None

            # Company Location
            try:
                company_location = company_div.find('div', {'class','css-1restlb eu4oa1w0'}).text.strip()
            except Exception:
                company_location = None

            # Salary Range
            try:
                metadata_div = td_tag.find('div', {'class','jobMetaDataGroup css-qspwa8 eu4oa1w0'}).ul
                salary_range = metadata_div.find('li', {'class','css-u74ql7 eu4oa1w0'}).div.div.text.strip()
            except AttributeError:
                salary_range = None

            # Summary
            try:
                summary_ul = card.find('div', {'class', 'underShelfFooter'}).div.div.ul
                summary_items = summary_ul.find_all('li')
                summary = " ".join([li.text.strip() for li in summary_items])
            except AttributeError:
                summary = None

            new_data = pd.DataFrame({
                'Job Title': [job_title],
                'Company Name': [company_name],
                'Location': [company_location],
                'Salary Range': [salary_range],
                'Summary': [summary],
                'Link': [job_url]
            })
            df = pd.concat([df, new_data], ignore_index=True)

        # Move to next page
        try:
            url = indeed_url + soup.find('a', {'aria-label':'Next Page'}).get('href')
        except AttributeError:
            break

Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found
Text Found


In [17]:
df

,Job Title,Company Name,Location,Salary Range,Summary,Link
0,,,,,,
1,Jr./Sr. Data Analyst (We Provide Training) - For Philippines Only,Kooapps Philippines Corporation,Philippines,None,Work with large amounts of data to identify gaming experience insights. Provide innovative ways to better understand correlation of data.,https://ph.indeed.com/rc/clk?jk=2e0c06bcebf076d4&bb=f2_xSJgK1DwHb1dJk_6a9FhaX51fDdojeb3r2DkyatZayRpKXiiSMfLyVSe9sv_0K2vMF38goJ12nV25aBT5A7L3Yi6Czvbg-8i3F2rt1tP4-uMA6c0Acu2r-EF89gcEpg15tP928-Y%3D&xkcb=SoCd67M3y7_-w4gNUb0LbzkdCdPP&fccid=90310e4cdefed57e&cmp=Kooapps-Philippines-Corporation&ti=Junior+Data+Analyst&vjs=3
2,Data Analyst,Maxicare Healthcare,Makati,None,Measure success through key performance indicators and other metrics. Provide analysis and meaningful information supporting decision-making and action-planning…,https://ph.indeed.com/rc/clk?jk=c9141baf87637ba2&bb=f2_xSJgK1DwHb1dJk_6a9K20zMqDWSbcWa1EG4NGdLSVZOGqpCa1MDK4QpswAm3Gzti7jfaOLJFMdHSWPnslayXUFsC0LQh1Ns_IyF0NUqN134v724r9nswEI1azNgVOGTuIXlgdbkc%3D&xkcb=SoAp67M3y7_-w4gNUb0KbzkdCdPP&fccid=6f952a954909a9dc&vjs=3
3,Data Analyst,S.P. Madrid & Associates,Parañaque,None,"The Data Analyst manages and assembles data from internal and external sources, ensuring accuracy and completeness.",https://ph.indeed.com/rc/clk?jk=b5a74714ca649c9b&bb=f2_xSJgK1DwHb1dJk_6a9PlaPAGNlwcGoNarWp1EmFwc_Ksru0HlGYSy_bRlPaXp383NCEi5ZcyxZ0U2hM4bcdpOEc9eBJWOpYdvuQQ5IEFo-2XksUSwsHGo9Bj10L1_eGGeVBRIiEQ%3D&xkcb=SoC067M3y7_-w4gNUb0JbzkdCdPP&fccid=355ac1fda32160ef&cmp=S.P.-Madrid-%2526-Associates&ti=Data+Analyst&vjs=3
4,Robotics Intern,Planate Management Group,Olongapo,None,"As a Robotics Intern, you will perform research related to trends and new practices in robotics industry and work closely with the Project Manager and the Lead…",https://ph.indeed.com/rc/clk?jk=e8c4581419bb381d&bb=f2_xSJgK1DwHb1dJk_6a9JAlHIyHVBGfT6u_vwyzlbmGuPb6tULPAc9RduzKMfQfqU9Rb2s_xZGDkb57UHAnIadroXuuollUCIOKH5n1sUnf9I9Y4wQ1app6egnE7dUMF1WKfEh7uxw%3D&xkcb=SoAA67M3y7_-w4gNUb0IbzkdCdPP&fccid=07812d1dcc5255e6&vjs=3
5,Software Design Engineer I,NEC Telecom Software Philippines,Taguig,None,Write functional and design specifications. Write original source codes and revise existing software. Design and implement software solutions.,https://ph.indeed.com/rc/clk?jk=ed86f468fa80b38a&bb=f2_xSJgK1DwHb1dJk_6a9JAlHIyHVBGfKxTzRKhYfcPESVOq9-X1uAqV0xLk_gV00SwEnRqvBNaNQeOK8wkSV3dOxmltpRD7FLzwK9kZ4R6ymmymKWxOoSoZwHsTpLxcA8iQf5uzgo0%3D&xkcb=SoCO67M3y7_-w4gNUb0PbzkdCdPP&fccid=cd106fcc81f9985b&cmp=NEC-Corporation&ti=User+Experience+Designer&vjs=3
6,Virtual Assistant: Associate Developer,Playbook,Remote in Manila,"PHP 19,000 - PHP 22,000 a month","In this role, you will need to have coding, python and javascript experience, . This position requires someone who can communicate clearly with clients and team…",https://ph.indeed.com/rc/clk?jk=447cef87a02e7b55&bb=f2_xSJgK1DwHb1dJk_6a9NMeaDqHtxmILwgbp328KiyqZfQyVglQ-faiTLRrfdvRZT_bmFwMER5m5zdYoFyt-sQo3gthwYGEPwupqfSr-ZRsCPWy2qIB12z6qhjtU7Pu-TuRSLes_EA%3D&xkcb=SoA667M3y7_-w4gNUb0ObzkdCdPP&fccid=2d8787d3cda2c9e2&cmp=Playbook-Sports&ti=Virtual+Assistant&vjs=3
7,Data Analyst | Data Engineer | Pioneer Team,iOPEX Phil,Taguig,None,"Collect, organize and analyze complex data sets from various sources. Identify trends, patterns and insights to support business objectives.",https://ph.indeed.com/rc/clk?jk=ddc8c35cc34b1cea&bb=f2_xSJgK1DwHb1dJk_6a9AtrWSxGeZZl4VPujjQbjWLmOpqSOxt_V6MMQ5wXBFE5CxX_2p0bRDKmkHqtrNouClTYusMwEQJ1_AJDwIcBfrFhUqcUftTajMtJdAmMy75RMWZMlzVoR9I%3D&xkcb=SoCn67M3y7_-w4gNUb0NbzkdCdPP&fccid=2ac70a6f96cda8fe&cmp=iEX-Tech-Ph&ti=Data+Analyst&vjs=3
8,Application Developer,INTEGRATED SECURITY AND AUTOMATION INC.,Mandaluyong Central Post Office 1550 P00,"PHP 22,000 - PHP 25,000 a month","Working with an agile team to develop, test, and maintain web and desktop-based multiplatform business applications leveraging on Open Source technologies.",https://ph.indeed.com/rc/clk?jk=04b1c6cb809ced9e&b

In [20]:
pd.set_option('display.max_colwidth', None)
df["Link"]

0                                                                                                                                                                                                                                                                                                                                      
1          https://ph.indeed.com/rc/clk?jk=2e0c06bcebf076d4&bb=f2_xSJgK1DwHb1dJk_6a9FhaX51fDdojeb3r2DkyatZayRpKXiiSMfLyVSe9sv_0K2vMF38goJ12nV25aBT5A7L3Yi6Czvbg-8i3F2rt1tP4-uMA6c0Acu2r-EF89gcEpg15tP928-Y%3D&xkcb=SoCd67M3y7_-w4gNUb0LbzkdCdPP&fccid=90310e4cdefed57e&cmp=Kooapps-Philippines-Corporation&ti=Junior+Data+Analyst&vjs=3
2                                                                     https://ph.indeed.com/rc/clk?jk=c9141baf87637ba2&bb=f2_xSJgK1DwHb1dJk_6a9K20zMqDWSbcWa1EG4NGdLSVZOGqpCa1MDK4QpswAm3Gzti7jfaOLJFMdHSWPnslayXUFsC0LQh1Ns_IyF0NUqN134v724r9nswEI1azNgVOGTuIXlgdbkc%3D&xkcb=SoAp67M3y7_-w4gNUb0KbzkdCdPP&fccid=6f952a954909a9dc&vjs=3
3               

In [22]:
df.to_csv("indeed_jobs.csv", index=True)

In [25]:
df.to_excel("indeed_jobs.xlsx", index=False)

# Jobstreet

In [ ]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
import math
from datetime import datetime, timedelta
import re

root_url = "https://ph.jobstreet.com"
job_title = "Python"
location = "Philippines"

rows = []

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

# First page
base_url = f"{root_url}/{job_title}-jobs/in-{location.replace(' ', '-')}"
url = base_url + "?page=1"
response = requests.get(url, impersonate="chrome")
soup = BeautifulSoup(response.text, 'lxml')

# Find total number of jobs
total_jobs_tag = soup.find("div", {"data-automation": "totalJobsMessage"})
if total_jobs_tag:
    total_jobs = int(total_jobs_tag.get_text(strip=True).split()[0].replace(",", ""))
else:
    total_jobs = 0

# Calculate last page
jobs_per_page = 32 # Check HTML
last_page = 1
# last_page = math.ceil(total_jobs / jobs_per_page)
print(f"Last Page: {last_page}")

for page in range(1, last_page + 1):
    page_url = base_url + f"?page={page}"
    print(f"Scraping: {page_url}")

    # Fetch HTML
    response = requests.get(url, impersonate="chrome")
    soup = BeautifulSoup(response.text, 'lxml')

    # Find all job cards
    job_cards = soup.find_all("article", {"data-card-type": "JobCard"})

    # Parse inside each card
    for card in job_cards:
        title_tag = card.find("a", {"data-automation": "jobTitle"})
        company_tag = card.find("a", {"data-automation": "jobCompany"})
        location_tags = card.find_all("a", {"data-automation": "jobLocation"})
        work_arrangement_tag = card.find("span", {"data-testid": "work-arrangement"})
        date_tag = card.find("span", {"data-automation": "jobListingDate"})

        title = title_tag.get_text(strip=True) if title_tag else None
        company = company_tag.get_text(strip=True) if company_tag else None
        locations = [loc.get_text(strip=True) for loc in location_tags] if location_tags else None
        work_arrangement = work_arrangement_tag.get_text(strip=True) if work_arrangement_tag else None
        
        # Find date
        text = date_tag.get_text(strip=True)
        match = re.search(r"\d+", text)
        if match:
            days_ago = int(match.group())
            date = (datetime.today() - timedelta(days=days_ago)).date()
            print(date)
        
        # Job details
        job_details_url = root_url + title_tag["href"]

        # Fetch job detail
        job_details_response = requests.get(job_details_url, impersonate="chrome")
        job_details_soup = BeautifulSoup(job_details_response.text, 'lxml')

        # Find job details
        job_details_tag = job_details_soup.find_all("div", {"data-automation": "jobAdDetails"})
        job_details = [details.get_text(strip=True) for details in job_details_tag]

        # Data Structuring
        rows.append({
            "Date Listed": date,
            "Job Title": title,
            "Company": company,
            "Location": locations,
            "Work Arrangement": work_arrangement,
            "Link": job_details_url,
            "Job Details": job_details
        })

df = pd.DataFrame(rows)

now = datetime.now()
today = now.strftime("%Y-%m-%d")
timestamp = now.strftime("%H%M%S")

title_filename = job_title.strip().replace(" ", "_")
excel_filename = f"{title_filename}_jobstreet_{today}_{timestamp}.xlsx"
df.to_excel(excel_filename, index=False)
print(f"Excel file: {excel_filename}")


Last Page: 1
Scraping: https://ph.jobstreet.com/Python-jobs/in-Philippines?page=1
2025-08-30
2025-08-30
2025-08-19
2025-08-15
2025-08-20
2025-08-30
2025-08-12
2025-08-21
2025-08-12
2025-08-27
2025-08-26
2025-08-13
2025-08-30
2025-08-13
2025-08-29
2025-08-15
2025-08-26
2025-08-07
2025-08-07
2025-08-20
2025-08-01
2025-08-29
2025-08-30
2025-08-02
2025-08-27
2025-08-21
2025-08-06
2025-08-23
2025-08-20
2025-08-16
2025-08-30
2025-08-29
Excel file: Python_jobstreet_2025-08-31_014627.xlsx
